# Version 3 — CNN-LSTM Fusion Model
Combines satellite image sequences with ERA5 weather and IBTrACS track features for advanced cyclone intensification prediction.

In [ ]:
!pip -q install tensorflow scikit-learn pandas numpy pillow matplotlib

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import zipfile
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, GlobalAveragePooling2D,
    LSTM, Dense, Dropout, Concatenate, TimeDistributed,
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)

print('TF version:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))

## Step 1: Upload Data Files
Upload:  
1. Your **Version 2 merged CSV** (containing IBTrACS + ERA5 columns).  
2. **satellite_images.zip** with images named `SID_YYYYMMDDHH.png` at 6-hour intervals.

In [ ]:
from google.colab import files

# ── Upload CSV ────────────────────────────────────────────────────────────────
print('Upload the merged CSV file:')
uploaded_csv = files.upload()
csv_path = list(uploaded_csv.keys())[0]
print(f'CSV: {csv_path}')

# ── Upload Satellite ZIP ───────────────────────────────────────────────────────
print('\nUpload satellite_images.zip:')
uploaded_zip = files.upload()
zip_path = list(uploaded_zip.keys())[0]

IMAGE_DIR = Path('/content/satellite_images')
IMAGE_DIR.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(IMAGE_DIR)
print(f'Extracted to {IMAGE_DIR}')

# Validate images directory
image_files = list(IMAGE_DIR.rglob('*.png'))
print(f'Found {len(image_files)} PNG files')
assert len(image_files) > 0, 'No PNG images found in zip!'

# Load CSV
df = pd.read_csv(csv_path, low_memory=False)
df['ISO_TIME'] = pd.to_datetime(df['ISO_TIME'], utc=True, errors='coerce')
df = df.sort_values(['SID', 'ISO_TIME']).reset_index(drop=True)
print(f'\nDataFrame shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

In [ ]:
# ── Sequence Configuration & Image Lookup ────────────────────────────────────
SEQUENCE_LENGTH = 4      # number of frames per sequence
STEP_HOURS      = 6      # hours between consecutive frames
IMAGE_SIZE      = (128, 128)

# Build lookup: stem (e.g. 'SID_YYYYMMDDHH') -> full path
image_lookup = {p.stem: str(p) for p in IMAGE_DIR.rglob('*.png')}
print(f'Image lookup entries: {len(image_lookup)}')
if image_lookup:
    print('  Sample keys:', list(image_lookup.keys())[:3])


def image_key(sid: str, ts) -> str:
    """Return the expected filename stem for a given SID and timestamp."""
    ts_pd = pd.Timestamp(ts)
    return f"{sid}_{ts_pd.strftime('%Y%m%d%H')}"


def get_sequence_paths(sid: str, ts, seq_len: int = SEQUENCE_LENGTH, step_h: int = STEP_HOURS):
    """
    Return a list of *seq_len* image paths ending at *ts*, spaced *step_h* hours apart.
    Returns None if any frame is missing.
    """
    ts_pd = pd.Timestamp(ts)
    paths = []
    for offset in range(-(seq_len - 1) * step_h, 1, step_h):
        frame_ts = ts_pd + pd.Timedelta(hours=offset)
        key = image_key(sid, frame_ts)
        if key not in image_lookup:
            return None
        paths.append(image_lookup[key])
    return paths  # length == seq_len


# Quick test
test_row = df.iloc[0]
test_paths = get_sequence_paths(test_row['SID'], test_row['ISO_TIME'])
print(f'\nSequence paths for first row: {test_paths}')

In [ ]:
# ── Create Labels ─────────────────────────────────────────────────────────────
def create_labels(df, start_hours, end_hours, threshold=15):
    df = df.copy().sort_values(['SID', 'ISO_TIME']).reset_index(drop=True)
    targets = np.full(len(df), np.nan)
    for sid, group in df.groupby('SID', sort=False):
        idxs  = group.index.tolist()
        times = group['ISO_TIME'].values
        winds = group['WIND'].values
        for pos, i in enumerate(idxs):
            t = times[pos]
            w_curr = winds[pos]
            dt_hours = (times - t) / np.timedelta64(1, 'h')
            mask = (dt_hours >= start_hours) & (dt_hours <= end_hours)
            future_winds = winds[mask]
            if len(future_winds) == 0:
                targets[i] = np.nan
            else:
                max_future = float(np.nanmax(future_winds))
                targets[i] = 1.0 if (max_future - w_curr) >= threshold else 0.0
    df['TARGET'] = targets
    return df


df_short = create_labels(df, start_hours=48,  end_hours=72)
df_long  = create_labels(df, start_hours=120, end_hours=168)
# Merge both target columns into one DataFrame
df['TARGET_SHORT'] = df_short['TARGET']
df['TARGET_LONG']  = df_long['TARGET']

print('Short-horizon targets:', df['TARGET_SHORT'].value_counts(dropna=False).to_dict())
print('Long-horizon  targets:', df['TARGET_LONG'].value_counts(dropna=False).to_dict())

In [ ]:
# ── Prepare Dataset ───────────────────────────────────────────────────────────
CANDIDATE_FEATURES = [
    'LAT', 'LON', 'WIND', 'MONTH', 'HOUR',
    'LAT_CHANGE', 'LON_CHANGE', 'WIND_CHANGE', 'TRACK_SPEED',
    'SST', 'MSL', 'U10', 'V10', 'T2M', 'D2M', 'TP',
]

# Add MONTH / HOUR if missing
if 'MONTH' not in df.columns:
    df['MONTH'] = df['ISO_TIME'].dt.month.astype(float)
if 'HOUR' not in df.columns:
    df['HOUR'] = df['ISO_TIME'].dt.hour.astype(float)

# Add diff features if missing
for col, src in [('LAT_CHANGE', 'LAT'), ('LON_CHANGE', 'LON'), ('WIND_CHANGE', 'WIND')]:
    if col not in df.columns:
        df[col] = df.groupby('SID')[src].diff()

FEATURE_COLS = [c for c in CANDIDATE_FEATURES if c in df.columns]
print(f'Tabular features ({len(FEATURE_COLS)}): {FEATURE_COLS}')

# Keep rows that have:
#   - complete image sequence
#   - non-null label for both horizons
valid_mask = (
    df['TARGET_SHORT'].notna() &
    df['TARGET_LONG'].notna() &
    df.apply(lambda r: get_sequence_paths(r['SID'], r['ISO_TIME']) is not None, axis=1)
)

df_valid = df[valid_mask].reset_index(drop=True)
print(f'\nRows with complete sequences AND labels: {len(df_valid)}')
print(f'Unique storms: {df_valid["SID"].nunique()}')
print(f'Short-horizon class dist: {df_valid["TARGET_SHORT"].value_counts().to_dict()}')
print(f'Long-horizon  class dist: {df_valid["TARGET_LONG"].value_counts().to_dict()}')

In [ ]:
# ── Build numpy Arrays ────────────────────────────────────────────────────────
def load_image(path: str) -> np.ndarray:
    """Load a PNG, convert to grayscale, resize to IMAGE_SIZE, normalise to [0,1]."""
    img = Image.open(path).convert('L').resize(IMAGE_SIZE)
    return np.array(img, dtype=np.float32)[..., np.newaxis] / 255.0  # (H, W, 1)


N = len(df_valid)
X_images   = np.zeros((N, SEQUENCE_LENGTH, IMAGE_SIZE[0], IMAGE_SIZE[1], 1), dtype=np.float32)
X_tabular  = df_valid[FEATURE_COLS].values.astype(np.float32)
y_short    = df_valid['TARGET_SHORT'].astype(np.float32).values
y_long     = df_valid['TARGET_LONG'].astype(np.float32).values
groups     = df_valid['SID'].values

print(f'Loading {N} image sequences …')
for i, row in df_valid.iterrows():
    paths = get_sequence_paths(row['SID'], row['ISO_TIME'])
    for t, p in enumerate(paths):
        X_images[i, t] = load_image(p)
    if i % 100 == 0:
        print(f'  {i}/{N}', end='\r')

print(f'\nX_images  shape: {X_images.shape}')
print(f'X_tabular shape: {X_tabular.shape}')
print(f'y_short   shape: {y_short.shape}')
print(f'y_long    shape: {y_long.shape}')

In [ ]:
# ── Train / Test Split & Scaling ──────────────────────────────────────────────
unique_storms = np.unique(groups)
assert len(unique_storms) >= 4, (
    f'Need at least 4 unique storms for GroupShuffleSplit, found {len(unique_storms)}.'
)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_tabular, y_short, groups))

X_img_train,  X_img_test  = X_images[train_idx],  X_images[test_idx]
X_tab_train,  X_tab_test  = X_tabular[train_idx], X_tabular[test_idx]
y_short_train, y_short_test = y_short[train_idx], y_short[test_idx]
y_long_train,  y_long_test  = y_long[train_idx],  y_long[test_idx]

# Impute then scale — fit only on training data
imputer = SimpleImputer(strategy='median')
scaler  = StandardScaler()
X_tab_train = scaler.fit_transform(imputer.fit_transform(X_tab_train))
X_tab_test  = scaler.transform(imputer.transform(X_tab_test))

print(f'Train: {len(train_idx)} rows | Test: {len(test_idx)} rows')
print(f'Train storms: {np.unique(groups[train_idx]).shape[0]}')
print(f'Test  storms: {np.unique(groups[test_idx]).shape[0]}')

# Validate both classes present
for name, y_tr, y_te in [('Short', y_short_train, y_short_test), ('Long', y_long_train, y_long_test)]:
    tr_cls = np.unique(y_tr[~np.isnan(y_tr)])
    te_cls = np.unique(y_te[~np.isnan(y_te)])
    print(f'{name} — train classes: {tr_cls}, test classes: {te_cls}')
    if len(tr_cls) < 2:
        print(f'  [WARNING] Only one class in {name} training set!')

In [ ]:
# ── Build CNN-LSTM Fusion Model ───────────────────────────────────────────────
def build_model(n_tabular_features: int) -> Model:
    """CNN-LSTM fusion model with dual output heads."""

    # ── Image branch: TimeDistributed CNN encoder → LSTM ─────────────────────
    img_input = Input(shape=(SEQUENCE_LENGTH, IMAGE_SIZE[0], IMAGE_SIZE[1], 1),
                      name='image_input')

    # CNN encoder applied to each frame independently
    x = TimeDistributed(Conv2D(32, (3, 3), activation='relu', padding='same'))(img_input)
    x = TimeDistributed(MaxPooling2D((2, 2)))(x)
    x = TimeDistributed(Conv2D(64, (3, 3), activation='relu', padding='same'))(x)
    x = TimeDistributed(MaxPooling2D((2, 2)))(x)
    x = TimeDistributed(Conv2D(128, (3, 3), activation='relu', padding='same'))(x)
    x = TimeDistributed(MaxPooling2D((2, 2)))(x)
    x = TimeDistributed(GlobalAveragePooling2D())(x)  # (batch, seq_len, 128)

    # LSTM reads the sequence of CNN embeddings
    x = LSTM(64, name='lstm_encoder')(x)              # (batch, 64)

    # ── Tabular branch ────────────────────────────────────────────────────────
    tab_input = Input(shape=(n_tabular_features,), name='tabular_input')
    t = Dense(64, activation='relu')(tab_input)
    t = Dropout(0.25)(t)

    # ── Fusion ────────────────────────────────────────────────────────────────
    merged = Concatenate(name='fusion')([x, t])
    merged = Dense(64, activation='relu')(merged)
    merged = Dropout(0.3)(merged)

    # ── Dual output heads ─────────────────────────────────────────────────────
    out_short = Dense(1, activation='sigmoid', name='risk_2_3_days')(merged)
    out_long  = Dense(1, activation='sigmoid', name='risk_5_7_days')(merged)

    model = Model(inputs=[img_input, tab_input],
                  outputs=[out_short, out_long])

    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss={
            'risk_2_3_days': 'binary_crossentropy',
            'risk_5_7_days': 'binary_crossentropy',
        },
        metrics={
            'risk_2_3_days': tf.keras.metrics.AUC(name='auc'),
            'risk_5_7_days': tf.keras.metrics.AUC(name='auc'),
        },
    )
    return model


model = build_model(n_tabular_features=X_tab_train.shape[1])
model.summary()

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.5, min_lr=1e-6, verbose=1),
]

history = model.fit(
    x=[X_img_train, X_tab_train],
    y={'risk_2_3_days': y_short_train, 'risk_5_7_days': y_long_train},
    validation_split=0.2,
    epochs=30,
    batch_size=8,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
# ── Evaluate ──────────────────────────────────────────────────────────────────
def evaluate_horizon(y_true, y_prob, label):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob).ravel()

    # Remove NaN labels (can occur if label creation had no future window)
    valid = ~np.isnan(y_true)
    y_true = y_true[valid].astype(int)
    y_prob = y_prob[valid]
    y_pred = (y_prob >= 0.5).astype(int)

    print(f'\n{"=" * 60}')
    print(f'  {label}')
    print(f'  n={len(y_true)} | class dist: {dict(zip(*np.unique(y_true, return_counts=True)))}')
    print('\n  Classification Report:')
    print(classification_report(y_true, y_pred, zero_division=0))
    print('  Confusion Matrix:')
    print(confusion_matrix(y_true, y_pred))

    n_classes = len(np.unique(y_true))
    if n_classes < 2:
        print('  [WARNING] Only one class present — skipping ROC/PR-AUC')
    else:
        print(f'  ROC-AUC : {roc_auc_score(y_true, y_prob):.4f}')
        print(f'  PR-AUC  : {average_precision_score(y_true, y_prob):.4f}')

    print(f'  Brier   : {brier_score_loss(y_true, y_prob):.4f}')


preds = model.predict([X_img_test, X_tab_test], batch_size=8)
pred_short = preds[0]  # risk_2_3_days
pred_long  = preds[1]  # risk_5_7_days

evaluate_horizon(y_short_test, pred_short, 'Short Horizon — 48–72 h (2–3 days)')
evaluate_horizon(y_long_test,  pred_long,  'Long Horizon  — 120–168 h (5–7 days)')

In [ ]:
# ── Training Loss Curve ───────────────────────────────────────────────────────
hist = history.history
epochs = range(1, len(hist['loss']) + 1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs, hist['loss'],     label='Train loss',      marker='o', markersize=3)
ax.plot(epochs, hist['val_loss'], label='Validation loss', marker='s', markersize=3, linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('V3 CNN-LSTM Training History')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Save & Download Model ─────────────────────────────────────────────────────
save_path = '/content/cyclone_v3.keras'
model.save(save_path)
print(f'Model saved to {save_path}')

from google.colab import files
files.download(save_path)
print('Download initiated.')